
Adapted from Gemini API Cookbook: https://github.com/google-gemini/cookbook/blob/main/examples/document_search.ipynb, see the original notebook which is under an Apache License.


# Chatting about AI music tools with LLMs, via document retrieval

<a href="https://colab.research.google.com/github/jasper-zheng/teaching/blob/main/AIAA_Fall2026/02-DocumentRetrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This example demonstrates how to use the Gemini API to create embeddings so that we can perform document search. We will build a word embedding that allows us to compare search strings, or questions, to document contents.

In this notebook, we'll use embeddings to perform document search over **four AI music tools** that are either not that popular, or very recent, so it's likely that a plain LLM won't know about them. We'll see how to use document retrieval to give LLMs this domain-specific knowledge.



## Setup


To set this notebook to the right environment, click the **Select kernel** button in the top right corner of this notebook, then select **Python Environments...**

To double check you have done this correctly, hit the run cell button (▶) on the cell below:

In [ ]:
import os
print(os.environ['CONDA_DEFAULT_ENV'])

In [ ]:
# Run this if you haven't installed the google-genai package yet
# %pip install google-genai

In [ ]:
import textwrap
import numpy as np
import pandas as pd

from google import genai
from google.genai import types

from IPython.display import Markdown

**Get a Google API key:** Go to [https://aistudio.google.com/api-keys](https://aistudio.google.com/api-keys) and hit "Create API key" at the top right. Name it anything you want. Select "Default Gemini Project" as your project. (Don't set up any billing!) Then you'll want to copy your API key (using the copy icon just to the right of the "set up billing" text) and paste it below:

In [ ]:
my_key = "__your_api_key_here__" #Please don't commit any code with your API key to github, or share your key with anyone

client = genai.Client(api_key=my_key)

### Select the model to be used

We'll use the `gemini-embedding-001` here. See a list of available embedding models here: https://ai.google.dev/gemini-api/docs/embeddings

Any embedding model will work, but for real applications it's important to choose a specific model and stick with it. The outputs of different models are not compatible with each other.

In [ ]:
MODEL_ID = "gemini-embedding-001"

## Generate the embeddings

In this section, you will see how to generate embeddings for the different texts in the dataframe using the embeddings from the Gemini API.

The Gemini embedding model supports several task types, each tailored for a specific goal. See a detailed list of tasks (e.g., classification, similarity search...) here: https://ai.google.dev/gemini-api/docs/embeddings#supported-task-types 

In [ ]:
sample_text = ("Title: Reflection Across AI-based Music Composition (Ford et al., 2024)"
    "\n"
    "Abstract:\n"
    "\n"
    "Reflection is fundamental to creative practice. However, the plurality of ways in which people reflect when using AI Generated Content (AIGC) is underexplored.")

embedding = client.models.embed_content(model=MODEL_ID,
                                contents=sample_text,
                                config=types.EmbedContentConfig(
                                  task_type="RETRIEVAL_DOCUMENT"))

print(embedding.embeddings)

## Building an embeddings database

Here are four pieces of text description about the four AI music tools. We'll use them to build the embeddings database. 

In [ ]:
DOCUMENT1 = {
    "title": "The Concatenator (Tralie and Cantil, 2024)",
    "content": "We present The Concatenator, a real time system for audio-guided concatenative synthesis. Similarly to Driedger et al.s musaicing (or audio mosaicing) technique, we concatenate a set number of windows within a corpus of audio to re-create the harmonic and percussive aspects of a target audio stream. Unlike Driedgers NMF-based technique, however, we instead use an explicitly Bayesian point of view, where corpus window indices are hidden states and the target audio stream is an observation. We use a particle filter to infer the best hidden corpus states in real-time. Our transition model includes a tunable parameter to control the time-continuity of corpus grains, and our observation model allows users to prioritize how quickly windows change to match the target. Because the computational complexity of the system is independent of the corpus size, our system scales to corpora that are hours long, which is an important feature in the age of vast audio data collections. Within The Concatenator module itself, composers can vary grain length, fit to target, and pitch shift in real time while reacting to the sounds they hear, enabling them to rapidly iterate ideas. To conclude our work, we evaluate our system with extensive quantitative tests of the effects of parameters, as well as a qualitative evaluation with artistic insights. Based on the quality of the results, we believe the real-time capability unlocks new avenues for musical expression and control, suitable for live performance and modular synthesis integration, which furthermore represents an essential breakthrough in concatenative synthesis technology."}
DOCUMENT2 = {
    "title": "Magenta RealTime (Google, 2025)",
    "content": "Magenta RT is the latest in a series of models and applications developed as part of the Magenta Project. It is the open-weights cousin of Lyria RealTime, the real-time generative music model powering Music FX DJ and the real-time music API in Google AI Studio, developed by Google DeepMind. Real-time music generation models open up unique opportunities for live music exploration and performance. As an open-weights model, Magenta RT is targeted towards eventually running locally on consumer hardware (currently runs on free-tier Colab TPUs). It is an 800 million parameter autoregressive transformer model trained on ~190k hours of stock music from multiple sources, mostly instrumental. "}
DOCUMENT3 = {
    "title": "Tölvera (Armitage et al., 2024)",
    "content": "Diverse intelligence is an emerging field that views intelligence as a spectrum, drawing insights from e.g. cell-based models and their evolution, and recognising the combined agential properties of biological and engineered materials across disciplines. Within diverse intelligence, basal cognition encapsulates the simpler end of the continuum, focusing on broadly applicable insights from the behaviour of single-celled organisms and simulation. Based on a desire for more diversity of real-time AI in NIME, we developed a library called Tolvera, initially for composable artificial life. In this paper we present Tolvera's design and the practicebased methodology that drove it, reviewing artistic works and emergent themes from design-practice iteration cycles. We reflect on how an early influence of artificial life gave way to an interest in reading T ̈olvera as a basal art medium, and how unexpected tendencies and capabilities play in perturbative aesthetic tension with compositional decisions. We describe how basal agency research, aesthetics and toolkits are influencing the direction of both design and practice, and review work-in-progress features. Finally, we reflect on how a cell's eye view of agency is shaping our thinking towards AI in NIME."}
DOCUMENT4 = {
    "title": "Latent terrain toolkit (Zheng et al., 2026)",
    "content": "Latent terrain is a tool to build corpus-based sound spaces/maps/materials to steer neural audio autoencoders/codecs (such as RAVE, Stable Audio Open (codec), Music2Latent). A terrain is a surface map for the autoencoder's latent space, taking coordinates in a control space as inputs, and producing continuous real-time latent trajectories that can be used for sound synthesis. Latent terrain aims to open up the creative possibilities of latent space navigation, allowing one to adapt the latent space of an autoencoder to easy-to-navigate interfaces (such as gestural controllers, stylus and tablets, XY-pads, and more), explore it like walking on a terrain surface, and build new musical instruments that compose and interact with AI audio generators."}
documents = [DOCUMENT1, DOCUMENT2, DOCUMENT3, DOCUMENT4]

df = pd.DataFrame(documents)
df.columns = ['Title', 'Text']
df

Get the embeddings for each of these bodies of text. Add this information to the dataframe.

In [ ]:
# Get the embeddings of each text and add to an embeddings column in the dataframe
def embed_fn(text):
  return client.models.embed_content(model=MODEL_ID,
                             contents=text,
                             config=types.EmbedContentConfig(
                               task_type="RETRIEVAL_DOCUMENT")
                             )

df['Embeddings'] = df.apply(lambda row: embed_fn(row['Text']).embeddings[0].values, axis=1)
df

## Document search with Q&A

Now that the embeddings are generated, let's create a Q&A system to search these documents. You will start with a query about AI music, create an embedding of the query, and compare it against the collection of embeddings in the dataframe.

The embedding of the question will be a vector (list of float values), which will be compared against the vector of the documents via similarity between two vectors. Read more about vector similarity here: https://www.pinecone.io/learn/vector-similarity/ 

Note, with the new embeddings model (`embedding-001`), specify the task type as `QUERY` for user query and `DOCUMENT` when embedding a document text.

Task Type | Description
---       | ---
RETRIEVAL_QUERY	| Specifies the given text is a query in a search/retrieval setting.
RETRIEVAL_DOCUMENT | Specifies the given text is a document in a search/retrieval setting.

In [ ]:
query = "Which one of these AI music tools can be connected to a gestural controller?"
model = MODEL_ID

request = client.models.embed_content(model=model,
                                contents=query,
                                config=types.EmbedContentConfig(
                                  task_type="RETRIEVAL_QUERY"))

Use the `find_best_passage` function to calculate the dot products, and then sort the dataframe from the largest to smallest dot product value to retrieve the relevant passage out of the database.

In [ ]:
def find_best_passage(query, dataframe):
  """
  Compute the distances between the query and each document in the dataframe
  using dot product.
  """
  query_embedding = client.models.embed_content(model=model,
                                                contents=query,
                                                config=types.EmbedContentConfig(
                                                  task_type="RETRIEVAL_QUERY")).embeddings[0].values
  dot_products = np.dot(np.stack(dataframe['Embeddings']), query_embedding)
  idx = np.argmax(dot_products)
  return dataframe.iloc[idx]['Text'] # Return text from index with max value

View the most relevant document from the database:

In [ ]:
passage = find_best_passage(query, df)
passage

## Question and Answering Application

Let's try to use the text generation API to create a Q & A system. Input your own custom data below to create a simple question and answering example. You will still use the dot product as a metric of similarity.

In [ ]:
def make_prompt(query, relevant_passage):
  escaped = relevant_passage.replace("'", "").replace('"', "").replace("\n", " ")
  prompt = textwrap.dedent("""You are a helpful and informative bot that answers questions using text from the reference passage included below. \
  Be sure to respond in a complete sentence, being comprehensive, including all relevant background information. \
  However, you are talking to a non-technical audience, so be sure to break down complicated concepts and \
  strike a friendly and converstional tone. Be encouraging in terms of prompting the user to explore the topic. \
  If the passage is irrelevant to the answer, you may ignore it.
  QUESTION: '{query}'
  PASSAGE: '{relevant_passage}'

    ANSWER:
  """).format(query=query, relevant_passage=escaped)

  return prompt

In [ ]:
prompt = make_prompt(query, passage)
print(prompt)

Choose one of the Gemini content generation models in order to find the answer to your query.

In [ ]:
answer = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

Markdown(answer.text)

## Tasks 

Try out some tasks that we've seen in the previous `02-FunctionCalling-BaristaBot.ipynb` notebook, For instance: 
 - Ask the LLM about a tool that is not in our database, see if it makes up content
 - Give it a personality by modifying the system prompts
 - Add some other documents to the database